<a href="https://colab.research.google.com/github/Prab999/metacritic-vs-emmy-tvshow-analysis/blob/main/DatabaseCreation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Database Architecture & Analytical Queries
This file is used to build out the PostgreSQL database architecture and execute relational queries to analyze the intersection of critical acclaim and Emmy success.

This notebook connects to a NEON PostgreSQL instance via SQLAlchemy. It establishes the `Shows` and `Awards` tables, enforces primary and foreign key constraints, and populates the database with the cleaned CSVs. The remainder of the notebook executes analytical SQL queries utilizing Common Table Expressions (CTEs), custom Views, and aggregations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings

warnings.filterwarnings('ignore')

In [2]:
!pip install psycopg2-binary sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.9 MB/s eta 0:00:00


In [3]:
from sqlalchemy import create_engine, text
from google.colab import userdata

In [4]:
#Read in datasets, connect to NEON Database
df_awards_table = pd.read_csv('awards.csv')
df_shows_table = pd.read_csv('shows.csv')

NEON_URL = userdata.get('NEON')

engine = create_engine(NEON_URL)

In [5]:
# Rename the column so it is all lowercase
df_shows_table.rename(columns={'releaseDate': 'releasedate'}, inplace=True)

# table definitions for postgreSQL
create_tables_sql = text("""
DROP TABLE IF EXISTS Awards CASCADE;
DROP TABLE IF EXISTS Shows CASCADE;

CREATE TABLE Shows (
    show_id INTEGER PRIMARY KEY,
    title TEXT,
    releasedate VARCHAR,
    rating VARCHAR,
    genres TEXT,
    metascore FLOAT,
    metascore_count INTEGER,
    userscore FLOAT,
    userscore_count INTEGER
);

CREATE TABLE Awards (
    show_id INTEGER,
    category TEXT,
    win BOOLEAN,
    year INTEGER,
    PRIMARY KEY (show_id, category, year),
    FOREIGN KEY (show_id) REFERENCES Shows(show_id)
);
""")

In [6]:
with engine.connect() as conn:
    conn.execute(create_tables_sql)
    conn.commit()
    print("Tables re-created successfully.")

# Import data
df_shows_table.to_sql('shows', engine, if_exists='append', index=False)
print("Shows data imported.")

df_awards_table.to_sql('awards', engine, if_exists='append', index=False)
print("Awards data imported.")

Tables re-created successfully.
Shows data imported.
Awards data imported.


In [ ]:
#get average ratings by users and by critics
query_1 = """
SELECT
    ROUND(AVG(metascore)::numeric, 2) AS average_critic_score,
    ROUND(AVG(userscore)::numeric, 2) AS average_user_score,
    COUNT(show_id) AS total_shows_analyzed
FROM shows;
"""

# Execute and display the result
display(pd.read_sql(query_1, engine))

,average_critic_score,average_user_score,total_shows_analyzed
0,68.87,62.66,687


In [ ]:
#average scores from both audience and users for shows that won and got
#nominated for emmys
query_2 = """
SELECT
    awards.win as emmy_winner,
    ROUND(AVG(shows.metascore)::numeric, 2) AS avg_critic_score,
    ROUND(AVG(shows.userscore)::numeric, 2) AS avg_user_score,
    COUNT(DISTINCT shows.show_id) AS number_of_shows
FROM shows
WHERE show_id IN (
    SELECT show_id FROM awards WHERE win = True
)
JOIN awards ON shows.show_id = awards.show_id
GROUP BY awards.win;
"""

display(pd.read_sql(query_2, engine))

,emmy_winner,avg_critic_score,avg_user_score,number_of_shows
0,False,73.15,72.68,666
1,True,75.32,75.23,329


In [ ]:
#Highest rated shows by critics to never win emmy
query_3 = """
SELECT
    s.title,
    s.metascore,
    s.userscore
FROM shows s
WHERE s.metascore > (
    -- Find the average critic score for shows that won an Emmy
    SELECT AVG(shows.metascore)
    FROM shows
    JOIN awards ON shows.show_id = awards.show_id
    WHERE awards.win = True
)
-- make sure the show has never won an Emmy
AND s.show_id NOT IN (
    SELECT show_id FROM awards WHERE win = True
)
ORDER BY s.metascore DESC
LIMIT 10;
"""

display(pd.read_sql(query_3, engine))

,title,metascore,userscore
0,My So-Called Life,92.0,84.0
1,The Wire,91.0,92.0
2,The Vietnam War,90.0,82.0
3,Boomtown,89.0,58.0
4,Longford,88.0,93.0
5,Better Things,88.0,78.0
6,Everybody Hates Chris,88.0,79.0
7,Brooklyn Bridge,88.0,82.0
8,Mr. Show with Bob and David,87.0,82.0
9,Catastrophe,87.0,80.0


In [ ]:
#lowest critics ratings to win emmy
query_4 = """
SELECT DISTINCT
    s.title,
    s.metascore,
    s.userscore
FROM shows s
JOIN awards a ON s.show_id = a.show_id
WHERE a.win = True
ORDER BY s.metascore ASC
LIMIT 10;
"""

display(pd.read_sql(query_4, engine))

,title,metascore,userscore
0,So You Think You Can Dance,31.0,65.0
1,Dinotopia,35.0,0.0
2,E-Ring,39.0,20.0
3,George Lopez,45.0,76.0
4,Late Night with Jimmy Fallon,48.0,66.0
5,Harry's Law,48.0,72.0
6,The Magnificent Seven,48.0,0.0
7,The Tonight Show with Jay Leno,48.0,36.0
8,The Kennedys,50.0,66.0
9,Heist,51.0,0.0


In [ ]:
#Creates view in database, adding sentiment values based on critic rating
create_view_sql = text("""
CREATE OR REPLACE VIEW show_sentiments AS
SELECT
    show_id,
    title,
    metascore,
    CASE
        -- sentiment categories
        WHEN metascore >= 80 THEN 'Universal Acclaim'
        WHEN metascore >= 60 THEN 'Generally Favorable'
        WHEN metascore >= 40 THEN 'Mixed'
        ELSE 'Poor'
    END AS critic_sentiment
FROM shows;
""")

with engine.connect() as conn:
    conn.execute(create_view_sql)
    conn.commit()
    print("View 'show_sentiments' created successfully!")

View 'show_sentiments' created successfully!


In [ ]:
#How many Emmy winnere fall into different categories created by the view
query_5 = """
SELECT
    v.critic_sentiment,
    COUNT(DISTINCT v.show_id) as number_of_emmy_winners
FROM show_sentiments v
JOIN awards a ON v.show_id = a.show_id
WHERE a.win = True
GROUP BY v.critic_sentiment
ORDER BY number_of_emmy_winners DESC;
"""

display(pd.read_sql(query_5, engine))

,critic_sentiment,number_of_emmy_winners
0,Generally Favorable,203
1,Universal Acclaim,91
2,Mixed,30
3,Poor,5


In [ ]:
#average rating for emmy winners, grouped by category

query_6 = """
SELECT
    a.category,
    ROUND(AVG(s.metascore)::numeric, 2) AS avg_winner_critic_score,
    ROUND(AVG(s.userscore)::numeric, 2) AS avg_winner_user_score,
    COUNT(DISTINCT s.show_id) AS number_of_winning_shows
FROM shows s
JOIN awards a ON s.show_id = a.show_id
WHERE a.win = True
  AND s.metascore IS NOT NULL
GROUP BY a.category
HAVING COUNT(DISTINCT s.show_id) >= 10
ORDER BY avg_winner_critic_score DESC;
"""

display(pd.read_sql(query_6, engine))

,category,avg_winner_critic_score,avg_winner_user_score,number_of_winning_shows
0,Outstanding Writing for a Drama Series,84.85,87.25,11
1,Outstanding Drama Series,81.19,84.10,16
2,Outstanding Lead Actor In A Drama Series,81.12,86.65,18
3,Outstanding Directing For A Comedy Series,80.96,80.65,18
4,Outstanding Writing for a Comedy Series,80.89,83.22,11
5,Outstanding Supporting Actor In A Drama Series,80.71,83.61,14
6,Outstanding Directing For A Drama Series,80.52,83.13,17
7,Outstanding Casting For A Drama Series,79.32,83.16,15
8,Outstanding Comedy Series,78.44,82.75,19
9,Outstanding Supporting Actor In A Comedy Series,78.35,82.08,13


In [ ]:
#Query created to check the total wins per emmy category in the data sets
#used to decide what categories provide more reliable data
query_temp = """
SELECT
    category,
    COUNT(show_id) AS total_wins_recorded
FROM awards
WHERE win = True
GROUP BY category
ORDER BY total_wins_recorded DESC
LIMIT 30;
"""

display(pd.read_sql(query_temp, engine))

,category,total_wins_recorded
0,Outstanding Comedy Series,36
1,Outstanding Drama Series,35
2,Outstanding Supporting Actress In A Comedy Series,34
3,Outstanding Lead Actor In A Comedy Series,32
4,Outstanding Supporting Actor In A Drama Series,31
5,Outstanding Lead Actress In A Comedy Series,30
6,Outstanding Lead Actor In A Drama Series,28
7,Outstanding Supporting Actress In A Drama Series,28
8,Outstanding Guest Actor In A Comedy Series,27
9,Outstanding Lead Actress In A Drama Series,26


In [ ]:
#Biggest upsets, largest score discrepency in winner vs nominee
query_7 = """
WITH PrestigeWinners AS (
    -- Isolate the winners in the top categories
    SELECT a.year, a.category, s.title AS winner_title, s.metascore AS winner_score
    FROM awards a
    JOIN shows s ON a.show_id = s.show_id
    WHERE a.win = True
      AND a.category IN ('Outstanding Drama Series', 'Outstanding Comedy Series', 'Outstanding Limited Series')
      AND s.metascore IS NOT NULL
),
PrestigeNominees AS (
    -- Isolate the losers in those exact same categories
    SELECT a.year, a.category, s.title AS nominee_title, s.metascore AS nominee_score
    FROM awards a
    JOIN shows s ON a.show_id = s.show_id
    WHERE a.win = False
      AND a.category IN ('Outstanding Drama Series', 'Outstanding Comedy Series', 'Outstanding Limited Series')
      AND s.metascore IS NOT NULL
)
-- Join them together and calculate the upset margin
SELECT
    w.year,
    w.category,
    w.winner_title,
    w.winner_score,
    n.nominee_title AS snubbed_nominee,
    n.nominee_score,
    (n.nominee_score - w.winner_score) AS upset_margin
FROM PrestigeWinners w
JOIN PrestigeNominees n ON w.year = n.year AND w.category = n.category
-- Only show instances where the loser had a higher score than the winner
WHERE n.nominee_score > w.winner_score
ORDER BY upset_margin DESC
LIMIT 15;
"""

display(pd.read_sql(query_7, engine))

,year,category,winner_title,winner_score,snubbed_nominee,nominee_score,upset_margin
0,2001,Outstanding Comedy Series,Sex and the City,64.0,Malcolm in the Middle,88.0,24.0
1,1993,Outstanding Drama Series,Picket Fences,60.0,Northern Exposure,83.0,23.0
2,1994,Outstanding Drama Series,Picket Fences,60.0,NYPD Blue,83.0,23.0
3,1994,Outstanding Drama Series,Picket Fences,60.0,Northern Exposure,83.0,23.0
4,1999,Outstanding Drama Series,The Practice,74.0,The Sopranos,94.0,20.0
5,2002,Outstanding Comedy Series,Friends,65.0,Curb Your Enthusiasm,84.0,19.0
6,2006,Outstanding Comedy Series,The Office,66.0,Curb Your Enthusiasm,84.0,18.0
7,1998,Outstanding Comedy Series,Frasier,79.0,The Larry Sanders Show,95.0,16.0
8,2000,Outstanding Drama Series,The West Wing,78.0,The Sopranos,94.0,16.0
9,2003,Outstanding Drama Series,The West Wing,78.0,The Sopranos,94.0,16.0


In [ ]:
#largest discrepency between critic and audience scores present in database
query_8 = """
SELECT
    s.title,
    s.metascore,
    s.userscore,
    ABS(s.metascore - s.userscore) AS score_discrepancy,
    CASE
        WHEN s.show_id IN (SELECT show_id FROM awards WHERE win = True) THEN True
        ELSE False
    END AS won_any_emmy
FROM shows s
WHERE s.userscore_count >= 100
  AND s.metascore IS NOT NULL
  AND s.userscore IS NOT NULL
ORDER BY score_discrepancy DESC
LIMIT 15;
"""

display(pd.read_sql(query_8, engine))

,title,metascore,userscore,score_discrepancy,won_any_emmy
0,Full Frontal with Samantha Bee,84.0,36.0,48.0,True
1,The Orville,36.0,83.0,47.0,False
2,The Conners,75.0,36.0,39.0,False
3,Stargate SG-1,48.0,85.0,37.0,False
4,Rules of Engagement,28.0,65.0,37.0,False
5,Star Trek: The Next Generation,51.0,86.0,35.0,True
6,The Larry Sanders Show,95.0,61.0,34.0,True
7,American Dad!,43.0,77.0,34.0,False
8,Babylon 5,55.0,88.0,33.0,True
9,Star Trek: Discovery,73.0,42.0,31.0,True


In [ ]:
#find middle year in database to use for additional query
query_find_middle_year = """
WITH MedianCalc AS (
    -- Find exact median year for winning shows
    SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY year) AS median_year
    FROM awards
    WHERE win = True
)
-- Tally the awards based on that median
SELECT
    CAST(m.median_year AS INTEGER) AS pivot_year,
    COUNT(CASE WHEN a.year < m.median_year THEN 1 END) AS awards_prior,
    COUNT(CASE WHEN a.year > m.median_year THEN 1 END) AS awards_post,
    COUNT(CASE WHEN a.year = m.median_year THEN 1 END) AS awards_during_pivot_year
FROM awards a
CROSS JOIN MedianCalc m
WHERE a.win = True
GROUP BY m.median_year;
"""

display(pd.read_sql(query_find_middle_year, engine))

,pivot_year,awards_prior,awards_post,awards_during_pivot_year
0,2008,782,794,68


In [ ]:
#Lists average score based on the era, 2008 being set as pivot
query_9 = """
WITH ShowEras AS (
    -- Assign each winning show to an era, no duplicates per era
    SELECT DISTINCT
        s.show_id,
        s.metascore,
        s.userscore,
        CASE
            WHEN a.year < 2008 THEN '1. Pre-2008'
            WHEN a.year > 2008 THEN '3. Post-2008'
            ELSE '2. Exactly 2008'
        END AS era
    FROM shows s
    JOIN awards a ON s.show_id = a.show_id
    WHERE a.win = True
      AND s.metascore IS NOT NULL
      AND s.userscore IS NOT NULL
)
-- Calculate true average of the shows in each era
SELECT
    era,
    ROUND(AVG(metascore)::numeric, 2) AS avg_critic_score,
    ROUND(AVG(userscore)::numeric, 2) AS avg_user_score,
    COUNT(show_id) AS distinct_shows_in_era
FROM ShowEras
GROUP BY era
ORDER BY era;
"""

display(pd.read_sql(query_9, engine))

,era,avg_critic_score,avg_user_score,distinct_shows_in_era
0,1. Pre-2008,70.80,67.05,143
1,2. Exactly 2008,70.76,67.70,33
2,3. Post-2008,73.82,70.79,205


In [ ]:
#Total emmy wins by genres, splits shows with multiple genres to be counted in each
query_10 = """
WITH ShowEmmyCounts AS (
    -- Get each unique show, its scores, and count its total Emmys before splitting
    SELECT
        s.show_id,
        s.genres,
        s.metascore,
        s.userscore,
        COUNT(a.show_id) AS total_emmys
    FROM shows s
    JOIN awards a ON s.show_id = a.show_id
    WHERE a.win = True
      AND s.metascore IS NOT NULL
      AND s.userscore IS NOT NULL
    GROUP BY s.show_id, s.genres, s.metascore, s.userscore
),
ExplodedGenres AS (
    -- Explode the genres. each show duplicated for multiple genres.
    -- EX: The Sopranos now becomes 1 row for Crime (with 21 Emmys)
    -- and 1 row for Drama (with 21 Emmys attached).
    SELECT
        TRIM(unnest(string_to_array(genres, ','))) AS single_genre,
        metascore,
        userscore,
        total_emmys
    FROM ShowEmmyCounts
)
-- Aggregate by single genre
SELECT
    single_genre,
    ROUND(AVG(metascore)::numeric, 2) AS pure_avg_critic_score,
    ROUND(AVG(userscore)::numeric, 2) AS pure_avg_user_score,
    SUM(total_emmys) AS total_emmys_won
FROM ExplodedGenres
GROUP BY single_genre
HAVING SUM(total_emmys) >= 10
ORDER BY total_emmys_won DESC;
"""

display(pd.read_sql(query_10, engine))

,single_genre,pure_avg_critic_score,pure_avg_user_score,total_emmys_won
0,Drama,73.28,69.15,1074.0
1,Comedy,72.82,71.04,659.0
2,Crime,75.45,76.17,320.0
3,Romance,70.21,63.77,290.0
4,Thriller,73.91,73.78,266.0
5,Mystery,72.35,75.60,255.0
6,Adventure,68.75,70.19,209.0
7,Action,69.83,69.33,202.0
8,Sci-Fi,67.97,73.17,138.0
9,History,72.00,57.89,128.0


In [ ]:
#Lists shows with most Emmy Wins, and scores for those shows
query_11 = """
SELECT
    s.title,
    s.metascore AS critic_score,
    s.userscore AS user_score,
    COUNT(a.show_id) AS total_emmy_wins
FROM shows s
JOIN awards a ON s.show_id = a.show_id
WHERE a.win = True
GROUP BY s.show_id, s.title, s.metascore, s.userscore
ORDER BY total_emmy_wins DESC
LIMIT 10;
"""

display(pd.read_sql(query_11, engine))

,title,critic_score,user_score,total_emmy_wins
0,Saturday Night Live,NaN,65.0,66
1,Game of Thrones,86.0,84.0,59
2,Frasier,79.0,85.0,37
3,Cheers,81.0,88.0,27
4,The Simpsons,87.0,79.0,26
5,The West Wing,78.0,82.0,26
6,Hill Street Blues,NaN,92.0,26
7,ER,79.0,78.0,23
8,Modern Family,87.0,83.0,22
9,The Sopranos,94.0,93.0,21
